<a href="https://colab.research.google.com/github/AnirudhPawar35/india-flight-delay-analytics/blob/main/notebooks/PurffleShorts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PurffleShorts in Google Colab: free AI YouTube Shorts generator

Make finished vertical videos for YouTube Shorts, TikTok and Instagram Reels in your browser, with nothing to
install on your computer. Give it a topic, a web page, or a long video to clip. It writes the script (any LLM),
voices it with free neural voices, adds footage, word-synced captions, music and sound effects, and renders
with ffmpeg.

**Run the cells from top to bottom.** The first demo needs no keys at all.

- Clipping long videos is faster on a GPU: *Runtime → Change runtime type → T4 GPU* (optional).
- Keys are read from Colab **Secrets** (the key icon on the left), never typed into the notebook.
- Source, docs and issues: [github.com/Chamanrajragu/purffle-shorts](https://github.com/Chamanrajragu/purffle-shorts) (a ⭐ helps other creators find it).

In [1]:
#@title 1. Install PurffleShorts (about a minute)
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:  # Colab already has ffmpeg and OpenCV (face framing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "purffle-shorts[clip] @ git+https://github.com/Chamanrajragu/purffle-shorts"], check=True)
os.makedirs("/content/purffle" if IN_COLAB else "purffle-colab", exist_ok=True)
os.chdir("/content/purffle" if IN_COLAB else "purffle-colab")
os.environ.setdefault("UPLOAD", "false")   # nothing is uploaded from this notebook
subprocess.run([sys.executable, "-m", "purffle_shorts", "--version"], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'purffle_shorts', '--version'], returncode=0)

In [2]:
#@title 2. A demo Short with no keys (sample script, free voice, generated backgrounds)
import glob

from IPython.display import Video, display

STYLE = "chat"  #@param ["facts", "chat", "dialogue", "reddit"]


def latest_video():
    files = sorted(glob.glob("output_videos/*/*.mp4"), key=os.path.getmtime)
    return files[-1] if files else None


subprocess.run([sys.executable, "-m", "purffle_shorts", "demo", "--style", STYLE], check=True)
display(Video(latest_video(), embed=True, width=320))

## Use your own AI model and footage

Add these in **Secrets** (key icon on the left) and switch on *Notebook access* for each:

| Secret | Where to get it | What for |
|---|---|---|
| `GEMINI_API_KEY` | [aistudio.google.com](https://aistudio.google.com/apikey) (free tier) | writes the script |
| or `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GROQ_API_KEY`, `OPENROUTER_API_KEY` | | any one LLM is enough |
| `PEXELS_API_KEY` | [pexels.com/api](https://www.pexels.com/api/) (free) | stock footage per scene |
| `PIXABAY_API_KEY` | [pixabay.com/api/docs](https://pixabay.com/api/docs/) (free) | more footage |

In [5]:
#@title 3. Load your keys from Colab Secrets
KEYS = ["GEMINI_API_KEY", "OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GROQ_API_KEY", "OPENROUTER_API_KEY",
        "DEEPSEEK_API_KEY", "PEXELS_API_KEY", "PIXABAY_API_KEY", "ELEVENLABS_API_KEY"]
try:
    from google.colab import userdata
    for k in KEYS:
        try:
            value = userdata.get(k)
        except Exception:
            value = None
        if value:
            os.environ[k] = value
            print("✔", k)
except ImportError:
    print("Not in Colab: using this machine's environment and .env")
subprocess.run([sys.executable, "-m", "purffle_shorts", "doctor"])

✔ GEMINI_API_KEY


CompletedProcess(args=['/usr/bin/python3', '-m', 'purffle_shorts', 'doctor'], returncode=1)

In [8]:
import os

os.environ["LLM_PROVIDER"] = "gemini"
os.environ["LLM_MODEL"] = "gemini-3.8-flash"

print("Gemini model updated:", os.environ["LLM_MODEL"])

Gemini model updated: gemini-3.8-flash


In [15]:
#@title 4. Make a Short about anything
TOPIC = "Nobody Is Coming to Save You"  #@param {type:"string"}
FORMAT = "story"  #@param ["auto", "facts", "story", "listicle", "myth", "quiz", "explainer", "dialogue", "chat", "reddit"]
LANGUAGE = "en"  #@param {type:"string"}
SECONDS = 35  #@param {type:"slider", min:15, max:90, step:5}
CAPTIONS = "bold"  #@param ["bold", "boxed", "neon", "clean", "karaoke", "minimal"]
subprocess.run([sys.executable, "-m", "purffle_shorts", "make", "--topic", TOPIC, "--style", FORMAT,
                "--lang", LANGUAGE, "--duration", str(SECONDS), "--caption-style", CAPTIONS, "--no-upload"],
               check=True)
display(Video(latest_video(), embed=True, width=320))

In [7]:
!python -m purffle_shorts make --topic "Nobody Is Coming to Save You" --style story --lang en --duration 35 --caption-style bold --no-upload

09:48:05 [INFO] Topic [manual]: Nobody Is Coming to Save You
09:48:05 [INFO] LLM: gemini:gemini-2.5-flash
09:48:05 [INFO] YouTube searches: nobody is coming to save you motivation; nobody is coming to save you i did enough; nobody is coming to save you david goggins; nobody is coming to save you get up
09:48:05 [WARNING] LLM gemini:gemini-2.5-flash failed: gemini:gemini-2.5-flash: HTTP 404 — [{   "error": {     "code": 404,     "message": "This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get
09:48:05 [ERROR] Video failed: All LLM providers failed:
  gemini:gemini-2.5-flash: gemini:gemini-2.5-flash: HTTP 404 — [{   "error": {     "code": 404,     "message": "This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the 

In [16]:
#@title 5. Clip a long video (podcast, interview, talk) into Shorts
#@markdown Upload a video with the folder icon on the left (or give a URL of a video you may reuse).
SOURCE = "https://www.youtube.com/watch?v=g9zGQdrAlMw&pp=ugUHEgVlbi1VUw%3D%3D"  #@param {type:"string"}
CLIPS = 3  #@param {type:"slider", min:1, max:8, step:1}
FRAMING = "face"  #@param ["face", "center", "blur"]
BRIEF = ""  #@param {type:"string"}
#@markdown BRIEF is optional, e.g. "the funniest moments" (needs an AI key).
cmd = [sys.executable, "-m", "purffle_shorts", "clip", SOURCE, "--count", str(CLIPS), "--crop", FRAMING,
       "--no-upload"] + (["--prompt", BRIEF] if BRIEF.strip() else [])
subprocess.run(cmd, check=True)
for f in sorted(glob.glob("output_videos/*/*.mp4"), key=os.path.getmtime)[-CLIPS:]:
    print(f)
    display(Video(f, embed=True, width=280))

CalledProcessError: Command '['/usr/bin/python3', '-m', 'purffle_shorts', 'clip', 'https://www.youtube.com/watch?v=g9zGQdrAlMw&pp=ugUHEgVlbi1VUw%3D%3D', '--count', '3', '--crop', 'face', '--no-upload']' returned non-zero exit status 1.

In [ ]:
#@title 6. Download everything you made
import shutil

archive = shutil.make_archive("purffle-videos", "zip", "output_videos")
if IN_COLAB:
    from google.colab import files
    files.download(archive)
else:
    print(archive)

**Next steps**

- Upload to YouTube automatically, schedule posts and run it every day on your own computer or a server:
  see the [README](https://github.com/Chamanrajragu/purffle-shorts#-quick-start).
- The Studio web app, the MCP server for Claude and Cursor, and the REST API for n8n run locally.

In [10]:
import os

print("LLM_PROVIDER:", os.getenv("LLM_PROVIDER"))
print("LLM_MODEL:", os.getenv("LLM_MODEL"))
print("GEMINI_MODEL:", os.getenv("GEMINI_MODEL"))

LLM_PROVIDER: gemini
LLM_MODEL: gemini-3.8-flash
GEMINI_MODEL: None


In [11]:
!python -m purffle_shorts make --topic "Nobody Is Coming to Save You" --style story --lang en --duration 35 --caption-style bold --no-upload

09:51:31 [INFO] Topic [manual]: Nobody Is Coming to Save You
09:51:31 [INFO] LLM: gemini:gemini-3.8-flash
09:51:31 [INFO] YouTube searches: nobody is coming to save you motivation; nobody is coming to save you i did enough; nobody is coming to save you david goggins; nobody is coming to save you get up
09:51:37 [WARNING] LLM gemini:gemini-3.8-flash attempt 1/2 failed: gemini:gemini-3.8-flash: HTTP 503 — [{   "error": {     "code": 503,     "message": "This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.",     "status": "UNAVAILABLE"   } } ] — retrying in 3.5s
09:51:49 [ERROR] LLM gemini:gemini-3.8-flash failed after 2 attempts: gemini:gemini-3.8-flash: HTTP 503 — [{   "error": {     "code": 503,     "message": "This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.",     "status": "UNAVAILABLE"   } } ]
09:51:49 [WARNING] LLM gemini:gemini-3.8-flash failed: gemini:gemi

In [12]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/gemini-4-carbon
models/lyri

In [13]:
import os

os.environ["LLM_PROVIDER"] = "gemini"
os.environ["LLM_MODEL"] = "gemini-3.5-flash-lite"

print("Using:", os.environ["LLM_MODEL"])

Using: gemini-3.5-flash-lite


In [14]:
!python -m purffle_shorts make --topic "Nobody Is Coming to Save You" --style story --lang en --duration 35 --caption-style bold --no-upload

09:53:44 [INFO] Topic [manual]: Nobody Is Coming to Save You
09:53:44 [INFO] LLM: gemini:gemini-3.5-flash-lite
09:53:44 [INFO] YouTube searches: nobody is coming to save you motivation; nobody is coming to save you i did enough; nobody is coming to save you david goggins; nobody is coming to save you get up
09:53:51 [INFO] Script (story, 105 words, 6 scenes): Nobody Is Coming To Save You
09:53:55 [INFO] Script Doctor rewrite scored 72, below the draft's 84: kept the draft
09:53:55 [INFO] Script Doctor: 84/100, kept — Cast array was empty instead of containing 'Narrator'.
09:54:41 [INFO] Voice: edge/en-US-AndrewMultilingualNeural, 42.1s, 105 words (native timing)
09:54:41 [INFO] Scene 1: no footage found for 'dense amazon rainforest aerial' — using an animated background
09:54:41 [INFO] Scene 2: no footage found for 'dense amazon rainforest aerial' — using an animated background
09:54:41 [INFO] Scene 4: no footage found for 'jungle floor shadows' — using an animated background
09:54:41 

In [17]:

from google.colab import userdata

for name in ["PEXELS_API_KEY", "PIXABAY_API_KEY"]:
    try:
        key = userdata.get(name)
        print(f"{name}: {'AVAILABLE' if key else 'EMPTY'}")
    except Exception:
        print(f"{name}: NOT ADDED")


PEXELS_API_KEY: NOT ADDED
PIXABAY_API_KEY: NOT ADDED


In [18]:

!python -m purffle_shorts clip "https://www.youtube.com/watch?v=g9zGQdrAlMw&pp=ugUHEgVlbi1VUw%3D%3D" --count 3 --crop face --no-upload


  Topic     ░░░░░░░░░░░░░░░░░░░░   2% · reading the videoWARNING: [youtube] No supported JavaScript runtime could be found. Only deno is enabled by default; to use another runtime add  --js-runtimes RUNTIME[:PATH]  to your command/config. YouTube extraction without a JS runtime has been deprecated, and some formats may be missing. See  https://github.com/yt-dlp/yt-dlp/wiki/EJS  for details on installing one
ERROR: [youtube] g9zGQdrAlMw: Sign in to confirm you’re not a bot. This helps protect our community. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/yt_dlp/YoutubeDL.py", line 1728, in wrapper
    return func(self, *args, **kwargs)
  File "

In [19]:

!pip install -U "yt-dlp[default]"
!pip install -U yt-dlp-ejs


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [mutagen]


In [20]:

!yt-dlp --version
!yt-dlp --simulate --no-warnings "https://www.youtube.com/watch?v=g9zGQdrAlMw"


2026.10.26
[youtube] Extracting URL: https://www.youtube.com/watch?v=g9zGQdrAlMw
[youtube] g9zGQdrAlMw: Downloading webpage
[youtube] g9zGQdrAlMw: Downloading visionos player API JSON
ERROR: [youtube] g9zGQdrAlMw: Sign in to confirm you’re not a bot. This helps protect our community. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


In [21]:
!deno --version

/bin/bash: line 1: deno: command not found


In [22]:
!curl -fsSL https://deno.land/install.sh | sh

######################################################################## 100.0%
Archive:  /root/.deno/bin/deno.zip
  inflating: /root/.deno/bin/deno    
Deno was installed successfully to /root/.deno/bin/deno
sh: 105: cannot open /dev/tty: No such device or address


In [23]:
import os

os.environ["PATH"] = "/root/.deno/bin:" + os.environ["PATH"]

!deno --version

deno 2.9.8 (stable, release, x86_64-unknown-linux-gnu)
v8 15.0.245.2-rusty
typescript 6.0.3


In [24]:
import os

os.environ["PATH"] = "/root/.deno/bin:" + os.environ["PATH"]

!yt-dlp --js-runtimes deno --simulate --verbose "https://www.youtube.com/watch?v=g9zGQdrAlMw"

[debug] Command-line config: ['--js-runtimes', 'deno', '--simulate', '--verbose', 'https://www.youtube.com/watch?v=g9zGQdrAlMw']
[debug] Encodings: locale utf-8, fs utf-8, pref utf-8, out utf-8, error utf-8, screen utf-8
[debug] yt-dlp version stable@2026.10.26 from yt-dlp/yt-dlp [c91f8de0b] (pip)
[debug] Python 3.13.16 (CPython x86_64 64bit) - Linux-6.6.159+-x86_64-with-glibc2.39 (OpenSSL 3.0.13 30 Jan 2024, glibc 2.39)
[debug] exe versions: ffmpeg 6.1.1 (setts), ffprobe 6.1.1
[debug] Optional libraries: Cryptodome-3.24.1, brotli-1.2.1, certifi-2026.10.23, curl_cffi-0.17.0, mutagen-1.48.1, requests-2.32.4, secretstorage-3.5.0, sqlite3-3.45.1, urllib3-2.5.0, websockets-15.0.1, yt_dlp_ejs-0.8.0
[debug] JS runtimes: deno-2.9.8
[debug] Proxy map: {'colab_language_server': '/usr/colab/bin/language_service'}
[debug] Request Handlers: urllib, requests, websockets, curl_cffi
[debug] Plugin directories: none
[debug] Loaded 1747 extractors
[debug] [youtube] [pot] PO Token Providers: none
[debug

In [25]:

!yt-dlp --js-runtimes deno \
  --extractor-args "youtube:player_client=tv" \
  --simulate \
  "https://www.youtube.com/watch?v=g9zGQdrAlMw"


[youtube] Extracting URL: https://www.youtube.com/watch?v=g9zGQdrAlMw
[youtube] g9zGQdrAlMw: Downloading webpage
[youtube] g9zGQdrAlMw: Downloading tv client config
[youtube] g9zGQdrAlMw: Downloading player a5c2aafe-main
[youtube] g9zGQdrAlMw: Downloading tv player API JSON
ERROR: [youtube] g9zGQdrAlMw: Sign in to confirm you’re not a bot. This helps protect our community. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
